# Homework 4: Evaluation Metrics for Classification

ML Zoomcamp 2026 — reproducible solution using the pinned official dataset.

## Setup and data

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mutual_info_score

DATA_URL = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(DATA_URL)
target = 'converted'
X = df.drop(columns=[target]).copy()
categorical = X.select_dtypes(include=['object']).columns.tolist()
numerical = X.select_dtypes(exclude=['object']).columns.tolist()
X[categorical] = X[categorical].fillna('NA')
X[numerical] = X[numerical].fillna(0.0)

def fit_model(X_train, y_train, X_val, C=1.0):
    dv = DictVectorizer(sparse=False)
    train_matrix = dv.fit_transform(X_train.to_dict(orient='records'))
    val_matrix = dv.transform(X_val.to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(train_matrix, y_train)
    return model, val_matrix


## Split

In [2]:
prepared = X.assign(converted=df.converted.to_numpy())
full_train, test = train_test_split(prepared, test_size=0.2, random_state=1)
train, val = train_test_split(full_train, test_size=0.25, random_state=1)
y_train=train.pop('converted'); y_val=val.pop('converted')

## Q1. Single-feature AUC

In [3]:
auc_by_feature = {}
for column in numerical:
    score = roc_auc_score(y_train, train[column])
    auc_by_feature[column] = max(score, 1-score)
candidates=['lead_score','number_of_courses_viewed','interaction_count','annual_income']
{c: auc_by_feature[c] for c in candidates}

{'lead_score': 0.7882254810906102, 'number_of_courses_viewed': 0.7229894623989618, 'interaction_count': 0.7649203047563227, 'annual_income': 0.6081977776250831}

**Answer:** `lead_score`.

## Q2. Model AUC

In [4]:
model, val_matrix = fit_model(train,y_train,val)
probabilities=model.predict_proba(val_matrix)[:,1]
validation_auc=roc_auc_score(y_val,probabilities)
validation_auc

0.7318222288173319

**Answer:** `0.732`.

## Q3–Q4. Precision/recall intersection and F1

In [5]:
thresholds=np.arange(0,1,0.01)
rows=[]
for threshold in thresholds:
    pred=(probabilities>=threshold).astype(int)
    precision=precision_score(y_val,pred,zero_division=0)
    recall=recall_score(y_val,pred,zero_division=0)
    rows.append((threshold,precision,recall,f1_score(y_val,pred,zero_division=0)))
metrics=pd.DataFrame(rows,columns=['threshold','precision','recall','f1'])
nonzero=metrics.loc[~((metrics.precision==0)&(metrics.recall==0))]
intersection=nonzero.loc[(nonzero.precision-nonzero.recall).abs().idxmin(),'threshold']
best_f1_threshold=metrics.loc[metrics.f1.idxmax(),'threshold']
intersection,best_f1_threshold

(np.float64(0.63), np.float64(0.41000000000000003))

**Answers:** Q3 `0.63`; Q4 `0.41`.

## Q5. Cross-validation variability

In [6]:
cv=KFold(n_splits=5,shuffle=True,random_state=1)
def cv_auc(C):
    scores=[]
    for train_idx,val_idx in cv.split(full_train):
        fold_train=full_train.iloc[train_idx].copy(); fold_val=full_train.iloc[val_idx].copy()
        fold_y=fold_train.pop('converted'); fold_val_y=fold_val.pop('converted')
        fold_model,fold_matrix=fit_model(fold_train,fold_y,fold_val,C=C)
        scores.append(roc_auc_score(fold_val_y,fold_model.predict_proba(fold_matrix)[:,1]))
    return scores
cv_scores=cv_auc(1.0)
np.mean(cv_scores),np.std(cv_scores)

(np.float64(0.7317269710972524), np.float64(0.006853233151022138))

**Answer:** `0.007`.

## Q6. Tune C

In [7]:
tuning={C:(np.mean(scores:=cv_auc(C)),np.std(scores)) for C in [1e-6,1e-3,1.0]}
tuning

{1e-06: (np.float64(0.6167342250714795), np.float64(0.018669799234831475)), 0.001: (np.float64(0.7491721776030379), np.float64(0.009996375455700753)), 1.0: (np.float64(0.7317269710972524), np.float64(0.006853233151022138))}

**Answer:** `0.001`.

## Checks

In [8]:
assert max(candidates,key=lambda c:auc_by_feature[c]) == 'lead_score'
assert round(validation_auc,3)==0.732
assert round(intersection,2)==0.63
assert round(best_f1_threshold,2)==0.41
assert round(np.std(cv_scores),3)==0.007
assert max(tuning,key=lambda c:(round(tuning[c][0],3),-round(tuning[c][1],3),-c))==0.001
print('All Homework 4 checks passed.')

All Homework 4 checks passed.
